In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/predictions_prompt_eval.csv"
df = pd.read_csv(csv_path)

df.head()

,idx,track_name,prompt_id,prompt_text,description
0,548,000787.mp3,p1_structured_closure,You are an expert music critic. Listen careful...,the track features a slow tempo and a focus on...
1,548,000787.mp3,p2_emotion_forward,Listen to the audio track and describe it prim...,the track creates a serene and introspective a...
2,548,000787.mp3,p3_sonic_detail,"Listen to the track and write a concise, factu...","the track features a slow tempo, around 90 bpm..."
3,548,000787.mp3,p4_user_query_style,Imagine a listener trying to describe this tra...,this track is a fusion of electronic and exper...
4,548,000787.mp3,p5_scene_imagery,Listen to the audio track and describe it by p...,the track creates a serene and introspective a...


In [ ]:
# Длина описаний в символах и словах
df['char_len'] = df['description'].astype(str).str.len()
df['word_len'] = df['description'].astype(str).str.split().str.len()

# Базовая статистика по каждому prompt_id
summary = df.groupby('prompt_id').agg(
    n_descriptions=('description', 'count'),
    mean_chars=('char_len', 'mean'),
    mean_words=('word_len', 'mean'),
    min_words=('word_len', 'min'),
    max_words=('word_len', 'max'),
)

print(summary)

                       n_descriptions  mean_chars  mean_words  min_words  \
prompt_id                                                                  
p1_structured_closure              20      585.40       94.85         51   
p2_emotion_forward                 20      424.95       67.20         49   
p3_sonic_detail                    20      423.05       67.80         45   
p4_user_query_style                20      437.15       72.95         52   
p5_scene_imagery                   20      466.10       76.30         58   

                       max_words  
prompt_id                         
p1_structured_closure        177  
p2_emotion_forward            88  
p3_sonic_detail              115  
p4_user_query_style          106  
p5_scene_imagery             114  


In [ ]:
for pid in df['prompt_id'].unique():
    print("\n=== Prompt:", pid, "===\n")
    sample = df[df['prompt_id'] == pid].sample(20, random_state=42)
    for _, row in sample.iterrows():
        print(f"- Track: {row.get('track_name', row.get('path', row['idx']))}")
        print(f"  Desc: {row['description']}\n")


=== Prompt: p1_structured_closure ===

- Track: 000787.mp3
  Desc: the track features a slow tempo and a focus on atmosphere and mood, with a general style or genre direction that leans towards experimental or avant-garde music. the rhythm is steady and consistent, with a moderate energy level that supports the overall mood. the dominant instruments are the piano and vocals, with the piano providing a steady, rhythmic foundation and the vocals adding a sense of emotion and depth.

- Track: 000590.mp3
  Desc: the song features a lively and upbeat tempo, with a bright and cheerful atmosphere. the general style is pop, with a focus on catchy melodies and accessible lyrics. the rhythm is energetic and danceable, with a strong emphasis on the beat. the dominant instruments are the piano and guitar, with the addition of a subtle bassline and percussion. the vocals are clear and expressive, with a focus on the melody and lyrics. overall, the song is a fun and uplifting piece that is sure to 

In [ ]:
import re
from itertools import combinations

def clean(text: str) -> set:
    """Простой препроцессинг: нижний регистр, убираем пунктуацию, берём множество слов."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return set(text.split())

def jaccard(a: str, b: str) -> float:
    """Jaccard-сходство между двумя строками по множеству слов."""
    sa, sb = clean(a), clean(b)
    if not sa or not sb:
        return 0.0
    inter = len(sa & sb)
    union = len(sa | sb)
    return inter / union

for pid in df['prompt_id'].unique():
    print("\n=== Prompt:", pid, "===\n")
    sample = df[df['prompt_id'] == pid].sample(20, random_state=42)

    # вывод примеров, как у тебя
    for _, row in sample.iterrows():
        print(f"- Track: {row.get('track_name', row.get('path', row['idx']))}")
        print(f"  Desc: {row['description']}\n")

    # считаем среднюю схожесть по парам внутри этого сэмпла
    descs = sample['description'].tolist()
    sims = []
    for i, j in combinations(range(len(descs)), 2):
        sims.append(jaccard(descs[i], descs[j]))

    mean_sim = sum(sims) / len(sims) if sims else 0.0
    print(f"> Mean Jaccard similarity for {pid} on this sample: {mean_sim:.3f}")


=== Prompt: p1_structured_closure ===

- Track: 000787.mp3
  Desc: the track features a slow tempo and a focus on atmosphere and mood, with a general style or genre direction that leans towards experimental or avant-garde music. the rhythm is steady and consistent, with a moderate energy level that supports the overall mood. the dominant instruments are the piano and vocals, with the piano providing a steady, rhythmic foundation and the vocals adding a sense of emotion and depth.

- Track: 000590.mp3
  Desc: the song features a lively and upbeat tempo, with a bright and cheerful atmosphere. the general style is pop, with a focus on catchy melodies and accessible lyrics. the rhythm is energetic and danceable, with a strong emphasis on the beat. the dominant instruments are the piano and guitar, with the addition of a subtle bassline and percussion. the vocals are clear and expressive, with a focus on the melody and lyrics. overall, the song is a fun and uplifting piece that is sure to 

In [ ]:


import pandas as pd

path = '/content/drive/MyDrive/data_with_descriptions_best_prompt.parquet'
df = pd.read_parquet(path)

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1230 entries, 0 to 1229
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   audio        1230 non-null   object
 1   title        1230 non-null   object
 2   artist       1230 non-null   object
 3   description  500 non-null    object
dtypes: object(4)
memory usage: 38.6+ KB


In [ ]:
path = '/content/drive/MyDrive/data.parquet'
df_orig = pd.read_parquet(path)


In [ ]:
df_orig.iloc[791]

,791
audio,{'bytes': b'ID3\x04\x00\x00\x00\x00\x052TIT2\x...
title,Blue Coochology
artist,Little Howlin' Wolf


In [ ]:
df.iloc[791]

,791
audio,{'bytes': b'ID3\x04\x00\x00\x00\x00\x052TIT2\x...
title,Blue Coochology
artist,Little Howlin' Wolf
description,None


In [ ]:
df = df[df['description'].notna()]
df

,audio,title,artist,description
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...,This World,AWOL,the audio features a laid-back and relaxed hip...
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...,Freeway,Kurt Vile,"the song features a mellow, laid-back vibe and..."
5,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05`TIT2\x...,Where is your Love?,Nicky Cook,the track exudes an overall tranquil and intro...
6,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05MTIT2\x...,Too Happy,Nicky Cook,"this song has a distinctly laid-back, atmosphe..."
10,{'bytes': b'ID3\x04\x00\x00\x00\x00\x03\x01TRC...,Father's Day,Abominog,"in this track, you can sense a complex, evolvi..."
...,...,...,...,...
1222,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05\nTIT2\...,The Laity,Sejayno,to better understand and interpret this track'...
1223,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04}TIT2\x...,Cemetary Rose,Sejayno,considering the track's use of ambient texture...
1226,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04}TIT2\x...,Vulcan's Hill,Sejayno,the song has a unique atmosphere and mood that...
1227,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04vTIT2\x...,Garden,Sejayno,despite its slow tempo and lack of consistent ...


In [ ]:
desc_col = 'description'

df = df[df['description'] != None]

df['desc_len'] = df[desc_col].astype(str).str.len()
df['word_count'] = df[desc_col].astype(str).str.split().str.len()

print(df['desc_len'].describe())
print(df['word_count'].describe())

print('Пустые описания:', (df[desc_col].isna() | (df[desc_col].astype(str).str.strip() == '')).sum())

count     500.000000
mean      504.534000
std       241.786663
min         9.000000
25%       362.250000
50%       515.500000
75%       650.500000
max      1349.000000
Name: desc_len, dtype: float64
count    500.000000
mean      77.866000
std       37.886869
min        2.000000
25%       56.000000
50%       79.000000
75%      100.250000
max      227.000000
Name: word_count, dtype: float64
Пустые описания: 0


In [ ]:
# Дубли
dup_count = df[desc_col].duplicated().sum()
print('Количество дублей описаний:', dup_count)

# Часто повторяющиеся фразы (очень грубо)
from collections import Counter

all_text = ' '.join(df[desc_col].astype(str).tolist()).lower()
for bad in ['n/a', 'no audio', 'test audio', 'sample track']:
    print(bad, 'встречается:', all_text.count(bad))

# Топ-20 самых длинных и самых коротких
print(df.sort_values('word_count').head(50)[[desc_col, 'word_count']])
print(df.sort_values('word_count', ascending=False).head(20)[[desc_col, 'word_count']])

Количество дублей описаний: 15
n/a встречается: 0
no audio встречается: 0
test audio встречается: 0
sample track встречается: 0
                                            description  word_count
677                                           no lyrics           2
1070                          minors, reverb, spacious.           3
1041                        **(1) atmosphere and mood**           4
1131                        **(1) atmosphere and mood**           4
210                         **(1) atmosphere and mood**           4
988                         **(1) atmosphere and mood**           4
1049                            [1] atmosphere and mood           4
1026                        **(1) atmosphere and mood**           4
560                  increase the volume progressively!           4
553                         **(1) atmosphere and mood**           4
679                  increase the volume progressively!           4
266                  increase the volume progressively! 

In [ ]:
df.sample(50)[[desc_col]].to_csv('descriptions_sample.csv', index=False)

In [ ]:
df[:50][[desc_col]].to_csv('descriptions_sample.csv', index=False)

In [ ]:
print(df.sort_values('word_count', ascending=False).iloc[450]['description'])

a track that features ambient atmospheric elements and lacks a distinct, catchy hook. the energy is generally low. instrumentation includes synths. the style is electronic.


In [ ]:
print(df.sort_values('word_count', ascending=False).iloc[400]['description'])

considering the electronic, techno style of the track, the general style and mood suggest a high-energy, lively atmosphere, characterized by rapid beats and dynamic electronic sounds. without specific information on vocals or instruments, the track's energy and dominance are mainly conveyed through the synthesisers, drum machines, and electronic effects.


In [ ]:
print(df.sort_values('word_count', ascending=False).iloc[250]['description'])

the track creates an immersive and engaging ambient experience. this piece evokes a sense of calm and introspection. instead of conventional elements like rhythmic patterns or beats, this ambient composition features evolving soundscapes and textures, designed to captivate the listener and evoke a reflective, meditative mood. without traditional vocals, the instrumental focus on synth and piano creates a hauntingly beautiful and meditative sound. through these textures and harmonies, the track showcases a high level of musicianship and technical mastery.


In [ ]:
print(df.sort_values('word_count', ascending=False).iloc[50]['description'])

this track features a laid-back, acoustic sound with a slow, melancholic energy that sets a contemplative mood. it employs simple, traditional instruments and showcases a raw, human quality. the overall vibe is one of emotional depth and introspection. the vocals are delivered with sincerity and convey a sense of vulnerability, further emphasizing the song's heartfelt nature. the track's style and instrumentation are reminiscent of folk, with elements of singer-songwriter and americana music. the use of acoustic instruments, such as guitar and mandolin, contributes to a warm, inviting atmosphere that invites the listener to focus on the lyrics and melody. overall, this track is a timeless example of folk music's capacity to evoke deep emotions and connect listeners to their shared cultural heritage.


In [ ]:
print(df.sort_values('word_count', ascending=False).iloc[15]['description'])

the song creates an ethereal and atmospheric vibe, with a spacious and open quality that makes you feel as though you are in a large, open space. the ambient genre is evident in the use of layered, reverb-heavy electronic sounds, creating a sense of space and depth. the music feels energetic, with a fast tempo and a sense of urgency that encourages movement and engagement. in terms of style, the track leans towards electronic music with a focus on atmospheric textures and soundscapes. in terms of instruments, the track relies heavily on synthesizers and electronic sounds, with some use of effects and sampling to create depth and movement. vocals, if present, would be ethereal and dreamy, contributing to the overall atmospheric feel of the track. overall, this song is a sonic journey that encourages movement, exploration, and a sense of freedom from the constraints of everyday life.


In [ ]:
print(df.sort_values('word_count', ascending=False).iloc[3]['description'])

this track is characterized by its atmospheric and moody sound, with a significant amount of space and dynamics creating an ambient and relaxing sonic landscape. the use of ambient and noise elements, as well as the absence of a clear melodic or rhythmic structure, lends itself to a general style or genre of experimental and avant-garde music. the general feel and mood of the track is that of energy and movement, as if it was created with the intent of getting the listener to explore their mind and body in a sensory and imaginative way. the dominant sound textures are achieved through the use of various ambient and noise elements, as well as synthesizers and electronic effects. the lack of a clear melody or rhythm gives the track a somewhat mysterious and exploratory quality, which makes it more effective at eliciting a specific emotional response from the listener. in terms of vocals, the absence of a clear melody or rhythm makes it more challenging to hear vocals in this track. witho

In [ ]:
df.head()

,audio,title,artist,description,desc_len,word_count
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...,This World,AWOL,the audio features a laid-back and relaxed hip...,714,118
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...,Freeway,Kurt Vile,"the song features a mellow, laid-back vibe and...",588,92
5,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05`TIT2\x...,Where is your Love?,Nicky Cook,the track exudes an overall tranquil and intro...,535,81
6,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05MTIT2\x...,Too Happy,Nicky Cook,"this song has a distinctly laid-back, atmosphe...",798,127
10,{'bytes': b'ID3\x04\x00\x00\x00\x00\x03\x01TRC...,Father's Day,Abominog,"in this track, you can sense a complex, evolvi...",419,71


In [ ]:
df_filtered = (
    df.sort_values("desc_len", ascending=True)
      .iloc[100:450]
)



df_filtered.to_parquet(
    "cut_descriptions.parquet",
    index=True
)

print(f"Было строк: {len(df)}")
print(f"Осталось строк: {len(df_filtered)}")
print(f"Минимальная длина после фильтра: {df_filtered['desc_len'].min()}")

Было строк: 500
Осталось строк: 350
Минимальная длина после фильтра: 325


In [ ]:
from pathlib import Path
save_dir = Path("/content/drive/MyDrive")
save_dir.mkdir(parents=True, exist_ok=True)

# Путь нового датасета
save_path = save_dir / "cut_descriptions.parquet"

df_filtered.to_parquet(save_path, index=True)

print(f"Было строк: {len(df):,}")
print(f"Осталось строк: {len(df_filtered):,}")
print(f"Минимальный desc_len после фильтра: {df_filtered['desc_len'].min()}")
print(f"Сохранено: {save_path}")

Было строк: 500
Осталось строк: 350
Минимальный desc_len после фильтра: 325
Сохранено: /content/drive/MyDrive/cut_descriptions.parquet


In [ ]:
df1 = pd.read_parquet("cut_descriptions.parquet")

In [ ]:
df1.head()

,audio,title,artist,description,desc_len,word_count
670,{'bytes': b'ID3\x04\x00\x00\x00\x00\x01_TXXX\x...,I Agree With The Access,Jad Fair,the track creates an atmospheric and experimen...,325,45
594,{'bytes': b'ID3\x04\x00\x00\x00\x00\x01BTCON\x...,Explosions,Heroin UK,the song has a high-energy feel and is designe...,327,56
1166,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02:TCON\x...,The Redvidier Box,Ric Royer,(a) a track with a strong atmospheric and intr...,327,51
486,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...,Evil Twin,Fuzz Unlimited,the song creates a haunting and reflective moo...,330,50
141,{'bytes': b'ID3\x04\x00\x00\x00\x00\x02STIT2\x...,Central Park,Blah Blah Blah,describing a track that emphasizes atmospheric...,331,47


In [ ]:
df1.describe()

,desc_len,word_count
count,400.000000,400.00000
mean,591.702500,91.33750
std,180.143632,28.66772
min,325.000000,45.00000
25%,457.750000,71.00000
50%,566.000000,86.00000
75%,693.250000,108.00000
max,1349.000000,227.00000


In [ ]:
print(df1.iloc[2]['description'])

(a) a track with a strong atmospheric and introspective vibe, reflecting its german and post-dubstep genre. it has a generally energetic and upbeat mood, with dynamic chord changes that contribute to its unique sound. the lack of vocals allows the listener to fully immerse themselves in the ambient and instrumental landscape.


In [2]:
import pandas as pd

csv_path = "/content/drive/MyDrive/sintetic_intents.csv"
df = pd.read_csv(csv_path)

df.head()

,idx,track_name,prompt_id,prompt_text,description
0,670,000921.mp3,q_mood,Listen to the track. Write one realistic music...,"considering the track's energetic tempo, the u..."
1,670,000921.mp3,q_scene,Listen to the track. Write one realistic music...,considering the track's experimental and avant...
2,670,000921.mp3,q_sound,Listen to the track. Write one realistic music...,"a track with a slow tempo and a deep, resonant..."
3,594,000840.mp3,q_mood,Listen to the track. Write one realistic music...,"given the track's energetic and upbeat mood, h..."
4,594,000840.mp3,q_scene,Listen to the track. Write one realistic music...,considering the energetic and driving nature o...


In [3]:
# Длина описаний в символах и словах
df['char_len'] = df['description'].astype(str).str.len()
df['word_len'] = df['description'].astype(str).str.split().str.len()

# Базовая статистика по каждому prompt_id
summary = df.groupby('prompt_id').agg(
    n_descriptions=('description', 'count'),
    mean_chars=('char_len', 'mean'),
    mean_words=('word_len', 'mean'),
    min_words=('word_len', 'min'),
    max_words=('word_len', 'max'),
)

print(summary)

           n_descriptions  mean_chars  mean_words  min_words  max_words
prompt_id                                                              
q_mood                175  164.885714   27.085714          9         54
q_scene               175  243.737143   39.668571          7         92
q_sound               175  210.005714   35.434286          7        197


In [4]:
for pid in df['prompt_id'].unique():
    print("\n=== Prompt:", pid, "===\n")
    sample = df[df['prompt_id'] == pid].sample(20, random_state=42)
    for _, row in sample.iterrows():
        print(f"- Track: {row.get('track_name', row.get('path', row['idx']))}")
        print(f"  Desc: {row['description']}\n")


=== Prompt: q_mood ===

- Track: 000462.mp3
  Desc: given the track's energetic tempo, use of brass instruments, and overall upbeat feel, a realistic music-search question about its mood and energy would be: what is the mood and energy of this track?

- Track: 001178.mp3
  Desc: considering the track's energetic and upbeat mood, how would you describe its overall feeling?

- Track: 000973.mp3
  Desc: considering the track's energetic tempo, the use of electronic sounds, and the overall upbeat and lively mood, a realistic music-search question about its mood and energy could be: what is the emotional impact of this track on the listener?

- Track: 000800.mp3
  Desc: considering the track's energetic tempo, driving rhythm, and the use of electronic sounds, i would like to know how these elements contribute to the overall mood and atmosphere of the music.

- Track: 000911.mp3
  Desc: considering the track's energetic tempo, driving rhythm, and overall upbeat mood, i would like to know ho